In [ ]:
import sys
from pathlib import Path
import azure.ai.evaluation

print("Python version:", sys.version.split()[0])
print("Python executable:", sys.executable)
print("Project directory:", Path.cwd())
print("Azure AI Evaluation: Available")

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Locate project files
project_dir = Path.cwd()
env_path = project_dir / ".env"
path = str(project_dir / "electronics_return_policy_ai_qa.jsonl")

# Load environment variables
load_dotenv(env_path)

# Azure OpenAI model configuration
model_config = {
    "azure_endpoint": os.getenv("AZURE_OPENAI_ENDPOINT"),
    "api_key": os.getenv("AZURE_OPENAI_API_KEY"),
    "azure_deployment": os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    "api_version": os.getenv("AZURE_OPENAI_API_VERSION")
}

# Azure AI Foundry project configuration
azure_ai_project = {
    "subscription_id": os.getenv("AZURE_SUBSCRIPTION_ID"),
    "resource_group_name": os.getenv("AZURE_OPENAI_RESOURCE_GROUP"),
    "project_name": os.getenv("AZURE_AI_PROJECT_NAME")
}

# Verify configuration
print("Dataset found:", Path(path).is_file())
print(".env found:", env_path.is_file())
print("Model configured:", all(model_config.values()))
print("Foundry project configured:", all(azure_ai_project.values()))
print("Deployment:", model_config["azure_deployment"])

In [ ]:
from azure.ai.evaluation import (
    GroundednessEvaluator,
    RelevanceEvaluator,
    CoherenceEvaluator,
    FluencyEvaluator,
    SimilarityEvaluator,
    F1ScoreEvaluator,
    RougeScoreEvaluator,
    RougeType,
    BleuScoreEvaluator,
    MeteorScoreEvaluator,
    GleuScoreEvaluator
)

print("All evaluator classes imported successfully!")

In [ ]:
# AI-assisted evaluators (GPT-4o)

groundedness_eval = GroundednessEvaluator(model_config)

relevance_eval = RelevanceEvaluator(model_config)

coherence_eval = CoherenceEvaluator(model_config)

fluency_eval = FluencyEvaluator(model_config)

similarity_eval = SimilarityEvaluator(model_config)

print("5 AI-assisted evaluators initialized!")

In [ ]:
# NLP-based evaluators

f1_eval = F1ScoreEvaluator()

rouge_eval = RougeScoreEvaluator(
    rouge_type=RougeType.ROUGE_1
)

bleu_eval = BleuScoreEvaluator()

meteor_eval = MeteorScoreEvaluator(
    alpha=0.9,
    beta=3.0,
    gamma=0.5
)

gleu_eval = GleuScoreEvaluator()

print("5 NLP evaluators initialized!")

In [ ]:
evaluators = {
    "groundedness": groundedness_eval,
    "relevance": relevance_eval,
    "coherence": coherence_eval,
    "fluency": fluency_eval,
    "similarity": similarity_eval,
    "f1_score": f1_eval,
    "rouge": rouge_eval,
    "bleu": bleu_eval,
    "meteor": meteor_eval,
    "gleu": gleu_eval
}

print("Total evaluators:", len(evaluators))
print("Evaluators ready:")

for name in evaluators:
    print("-", name)

In [ ]:
from pathlib import Path

print("Dataset found:", Path(path).is_file())
print("Total evaluators:", len(evaluators))
print("Foundry project configured:", bool(azure_ai_project))

print("\nEvaluators:")
for name in evaluators:
    print("-", name)

In [ ]:
from azure.ai.evaluation import evaluate

result = evaluate(
    data=path,

    evaluators=evaluators,

    evaluator_config={
        "default": {
            "column_mapping": {
                "query": "${data.query}",
                "response": "${data.response}",
                "context": "${data.context}",
                "ground_truth": "${data.ground_truth}"
            }
        }
    },

    azure_ai_project=azure_ai_project,

    evaluation_name="Electronics Store QA - All 10 Metrics",

    output_path="electronics_all_metrics_results.json"
)

print("Evaluation completed!")
print("Foundry URL:", result.get("studio_url"))